# PayShift India — 01 Data Cleaning

**Goal:** prepare two official datasets for analysing India's shift from cash and cards to UPI in **calendar year 2024**.

| Source | What it contains | Level |
|---|---|---|
| **NPCI** monthly product statistics (via Kaggle) | Transaction volumes for UPI, IMPS, NFS (ATM network), AePS, NACH, NETC | National, monthly |
| **RBI** Bank-wise ATM/POS/Card Statistics | ATMs, PoS terminals, QR codes, cards issued, card transactions, ATM cash withdrawals | Bank, monthly |

This notebook documents each step: what I inspected, what I found, and what I decided to do about it.

**Outputs** (saved to `data/processed/`): `npci_monthly_2024.csv`, `rbi_bank_cards_2024.csv`

## 1. Setup

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:,.2f}'.format

project = Path.cwd().parent            # notebook/ → payshift-india/
raw_dir = project / 'data' / 'raw'
processed_dir = project / 'data' / 'processed'
npci_dir = raw_dir / 'npci'
rbi_dir = raw_dir / 'rbi_cards'

print("Project folder:", project)
print("NPCI files:", len(list(npci_dir.glob('*.csv'))))
print("RBI files:", sorted(p.name for p in rbi_dir.iterdir()))

Project folder: c:\Users\anupa\OneDrive\Desktop\payshift-india
NPCI files: 11
RBI files: ['2024_01.XLSX', '2024_02.XLSX', '2024_03.XLSX', '2024_04.XLSX', '2024_05.XLSX', '2024_06.XLSX', '2024_07.XLSX', '2024_08.XLSX', '2024_09.XLSX', '2024_10.XLSX', '2024_11.XLSX', '2024_12.XLSX', '2026_08.xlsx']


---
# Part A: NPCI monthly statistics

## A1. Inspect one file first
Before combining 11 files, I looked at one (UPI) to understand the structure.

In [2]:
upi = pd.read_csv(npci_dir / 'UPI Monthly Product Statistics Trended.csv')
print("Shape:", upi.shape)
print(upi.dtypes)
display(upi.head())
display(upi.tail(3))

Shape: (12, 3)
Month                             str
Volume (in Mn.)                   str
Avg. Daily Volume (in Mn.)    float64
dtype: object


,Month,Volume (in Mn.),Avg. Daily Volume (in Mn.)
0,24-Jan,"12,203.02",393.65
1,24-Feb,"12,102.67",417.33
2,24-Mar,"13,440.00",433.55
3,24-Apr,"13,303.99",443.47
4,24-May,"14,035.84",452.77


,Month,Volume (in Mn.),Avg. Daily Volume (in Mn.)
9,24-Oct,"16,584.97",535.00
10,24-Nov,"15,482.02",516.07
11,24-Dec,"16,730.01",539.68


**Observations**
- Only **12 rows**: the data covers **January–December 2024**.
- It contains **volumes only** (number of transactions, in millions), not rupee values.
- Months are written as `24-Jan` (2-digit year, short month name), so they need converting to real dates.
- Volumes contain **thousand separators** (`12,203.02`), so pandas reads them as text unless told otherwise.

**Decision:** scope the national comparison to calendar 2024, the period this data covers.

## A2. Check that all files share the same columns
Stacking files only works if their columns match, so I checked every file's header.

In [3]:
for f in sorted(npci_dir.glob('*.csv')):
    d = pd.read_csv(f, nrows=2)
    print(f.name.replace(' Monthly Product Statistics Trended.csv', ''), '→', d.columns.tolist())

AePS - BHIM Aadhaar Pay → ['Month', 'Volume (in Mn.)', 'Avg. Daily Volume (in Mn.)']
AePS - Cash Withdrawal → ['Month', 'Volume (in Mn.)', 'Avg. Daily Volume (in Mn.)']
AePS - Funds Transfer → ['Unnamed: 0', 'Month', 'Volume (in Mn.)', 'Avg. Daily Volume (in Mn.)']
CTS → ['Month', 'Volume (in Mn.)', 'Avg. Daily Volume (in Mn.)']
IMPS → ['Month', 'Volume (in Mn.)', 'Avg. Daily Volume (in Mn.)']
NACH - APBS → ['Month', 'Volume (in Mn.)', 'Avg. Daily Volume (in Mn.)']
NACH - Credit → ['Month', 'Volume (in Mn.)', 'Avg. Daily Volume (in Mn.)']
NACH - Debit → ['Month', 'Volume (in Mn.)', 'Avg. Daily Volume (in Mn.)']
NETC → ['Month', 'Volume (in Mn.)', 'Avg. Daily Volume (in Mn.)']
NFS → ['Month', 'Volume (in Mn.)', 'Avg. Daily Volume (in Mn.)']
UPI → ['Month', 'Volume (in Mn.)', 'Avg. Daily Volume (in Mn.)']


**Observation:** *AePS – Funds Transfer* has an extra column, `Unnamed: 0`. This happens when a table is saved to CSV together with its row index. It carries no information.

**Decision:** drop any `Unnamed` column, and rename columns **by name** rather than by position, so an extra or reordered column can't silently misalign the data.

## A3. Combine all systems into one long table
Each file becomes rows tagged with a `system` label. A long format (one row per system per month) works well for SQL and Power BI filtering.

In [4]:
frames = []
for f in npci_dir.glob('*.csv'):
    d = pd.read_csv(f, thousands=',')                      # remove thousand separators
    d = d.loc[:, ~d.columns.str.startswith('Unnamed')]     # drop stray index column
    d['system'] = f.name.replace(' Monthly Product Statistics Trended.csv', '')
    frames.append(d)

npci = pd.concat(frames, ignore_index=True)
npci = npci.rename(columns={
    'Month': 'month',
    'Volume (in Mn.)': 'volume_mn',
    'Avg. Daily Volume (in Mn.)': 'avg_daily_volume_mn'
})
npci['month'] = pd.to_datetime(npci['month'], format='%y-%b')

print(npci.dtypes)
print("\nShape:", npci.shape)
print("Systems:", sorted(npci['system'].unique()))

month                  datetime64[us]
volume_mn                     float64
avg_daily_volume_mn           float64
system                            str
dtype: object

Shape: (132, 4)
Systems: ['AePS - BHIM Aadhaar Pay', 'AePS - Cash Withdrawal', 'AePS - Funds Transfer', 'CTS', 'IMPS', 'NACH - APBS', 'NACH - Credit', 'NACH - Debit', 'NETC', 'NFS', 'UPI']


**Check:** 11 systems × 12 months = 132 rows, `month` is a datetime, and both volume columns are numeric.

## A4. First look: how did each system change over 2024?

In [5]:
pivot = npci.pivot(index='month', columns='system', values='volume_mn')

growth = pd.DataFrame({
    'jan_2024_mn': pivot.iloc[0],
    'dec_2024_mn': pivot.iloc[-1],
})
growth['growth_pct'] = ((growth['dec_2024_mn'] / growth['jan_2024_mn'] - 1) * 100).round(1)
growth.sort_values('growth_pct', ascending=False)

,jan_2024_mn,dec_2024_mn,growth_pct
system,,,
NACH - Credit,94.81,131.59,38.80
UPI,"12,203.02","16,730.01",37.10
NACH - APBS,211.87,245.14,15.70
NETC,331.38,381.98,15.30
AePS - Cash Withdrawal,83.97,91.09,8.50
NFS,313.73,291.67,-7.00
CTS,54.72,50.66,-7.40
NACH - Debit,54.72,50.66,-7.40
AePS - BHIM Aadhaar Pay,1.88,1.72,-8.50


**Observations**
- **UPI grew ~37%** on a very large base, far ahead of every other system in scale.
- **IMPS fell ~13%**, suggesting person-to-person transfers are moving to UPI.
- **NFS (interbank ATM network) fell ~7%**, consistent with less cash use.
- **AePS cash withdrawal rose ~8.5%**: rural, Aadhaar-based cash demand is still growing, so the shift away from cash isn't uniform.
- *AePS – Funds Transfer* has a tiny base (under 0.05 Mn/month), so its percentage change isn't meaningful.

**Something looks wrong:** CTS and NACH – Debit show exactly the same January and December values. Two different payment systems shouldn't match to two decimal places.

## A5. Data quality check: CTS vs NACH – Debit

In [6]:
compare = pivot[['CTS', 'NACH - Debit']].copy()
compare['identical'] = compare['CTS'] == compare['NACH - Debit']
print("Identical in all 12 months:", compare['identical'].all())
compare

Identical in all 12 months: True


system,CTS,NACH - Debit,identical
month,,,
2024-01-01,54.72,54.72,True
2024-02-01,54.19,54.19,True
2024-03-01,58.53,58.53,True
2024-04-01,52.57,52.57,True
2024-05-01,52.44,52.44,True
2024-06-01,48.44,48.44,True
2024-07-01,53.10,53.10,True
2024-08-01,50.85,50.85,True
2024-09-01,48.49,48.49,True


**Finding:** the two systems are identical in **every month**. This is a data error in the source dataset, most likely one system's data copied into the other's file. It isn't possible to tell from this file which series is genuine.

**Decision:** exclude **both** rather than risk analysing incorrect data, and document it in the README.

In [7]:
npci = npci[~npci['system'].isin(['CTS', 'NACH - Debit'])]
print("Shape after exclusion:", npci.shape)
print("Systems kept:", sorted(npci['system'].unique()))

Shape after exclusion: (108, 4)
Systems kept: ['AePS - BHIM Aadhaar Pay', 'AePS - Cash Withdrawal', 'AePS - Funds Transfer', 'IMPS', 'NACH - APBS', 'NACH - Credit', 'NETC', 'NFS', 'UPI']


## A6. Save

In [8]:
npci.to_csv(processed_dir / 'npci_monthly_2024.csv', index=False)
print("Saved npci_monthly_2024.csv ✓")

Saved npci_monthly_2024.csv ✓


---
# Part B: RBI Bank-wise ATM/POS/Card Statistics

RBI publishes one Excel file per month. I downloaded January–December 2024 from rbi.org.in and named them `YYYY_MM` so the month can be read from the file name.

## B1. Inspect the raw layout of one file
RBI's Excel sheets have merged, multi-row headers, so I loaded one file with `header=None` to see the grid exactly as it is.

In [9]:
jan_path = next(p for p in rbi_dir.iterdir() if p.name.lower() == '2024_01.xlsx')
raw = pd.read_excel(jan_path, header=None)
print("Shape:", raw.shape)

# Print the header area row by row, skipping empty cells (merged headers leave many blanks)
for i in range(0, 12):
    print(f"Row {i}: {raw.iloc[i].dropna().tolist()}")

Shape: (106, 29)
Row 0: ['ATM, Acceptance Infrastructure and Card Statistics for the Month of January 2024']
Row 1: ['Sr. No.', 'Bank Name', 'Infrastructure ', 'Card Payments and Cash Withdrawal Transactions during the month']
Row 2: ['Number - Outstanding (as on month end)', 'Credit Card', 'Debit Card']
Row 3: ['ATMs & CRMs', 'PoS', 'Micro ATMs', 'Bharat QR Codes', 'UPI QR Codes', 'Credit Cards', 'Debit Cards', 'Card Payments Transactions', 'Cash Withdrawal', 'Card Payments Transactions', 'Cash Withdrawal']
Row 4: ['at PoS', 'Online (e-com)', 'Others', 'At ATM', 'at PoS', 'Online (e-com)', 'Others', 'ATM', 'PoS']
Row 5: ['On-site', 'Off-site', 'Volume (in actuals)', "Value (in Rs'000)", 'Volume (in actuals)', "Value (in Rs'000)", 'Volume (in actuals)', "Value (in Rs'000)", 'Volume (in actuals)', "Value (in Rs'000)", 'Volume (in actuals)', "Value (in Rs'000)", 'Volume (in actuals)', "Value (in Rs'000)", 'Volume (in actuals)', "Value (in Rs'000)", 'Volume (in actuals)', "Value (in Rs'00

**Observations**
- Column 0 is empty; **Sr. No.** and **Bank Name** are in columns 1 and 2.
- The headings are spread over several merged rows (Infrastructure → ATMs & CRMs → On-site / Off-site, and Credit Card → Card Payments → at PoS → Volume / Value).
- One row simply **numbers the data columns 1–26**. Instead of reconstructing the merged headers, I used these numbers to map each column to a clean name.

**Column map**

| No. | Name | Meaning | Type |
|---|---|---|---|
| 1–2 | `atm_onsite`, `atm_offsite` | ATMs & cash recyclers at / away from branches | Stock (month end) |
| 3 | `pos_terminals` | Card machines at merchants | Stock |
| 4 | `micro_atms` | Handheld devices used by banking agents | Stock |
| 5–6 | `bharat_qr`, `upi_qr` | QR codes deployed at merchants | Stock |
| 7–8 | `credit_cards`, `debit_cards` | Cards outstanding | Stock |
| 9–16 | `cc_*` | Credit card transactions: PoS, e-commerce, others, ATM cash (volume + value) | Flow (during month) |
| 17–26 | `dc_*` | Debit card transactions: PoS, e-commerce, others, ATM cash, PoS cash (volume + value) | Flow |

Volumes are actual counts; values are in **Rs '000**.

In [10]:
cols = ['sr_no', 'bank_name',
        'atm_onsite', 'atm_offsite', 'pos_terminals', 'micro_atms',
        'bharat_qr', 'upi_qr', 'credit_cards', 'debit_cards',
        'cc_pos_vol', 'cc_pos_val', 'cc_ecom_vol', 'cc_ecom_val',
        'cc_others_vol', 'cc_others_val', 'cc_atm_cash_vol', 'cc_atm_cash_val',
        'dc_pos_vol', 'dc_pos_val', 'dc_ecom_vol', 'dc_ecom_val',
        'dc_others_vol', 'dc_others_val', 'dc_atm_cash_vol', 'dc_atm_cash_val',
        'dc_pos_cash_vol', 'dc_pos_cash_val']
num_cols = cols[2:]

num_row = raw.index[raw.iloc[:, 3] == 1][0]      # the row that numbers the columns 1–26
print("Numbered header row:", num_row)

rbi = raw.iloc[num_row + 1:, 1:29].copy()
rbi.columns = cols
display(rbi.head(8))
display(rbi.tail(8))

Numbered header row: 6


,sr_no,bank_name,atm_onsite,atm_offsite,pos_terminals,micro_atms,bharat_qr,upi_qr,credit_cards,debit_cards,cc_pos_vol,cc_pos_val,cc_ecom_vol,cc_ecom_val,cc_others_vol,cc_others_val,cc_atm_cash_vol,cc_atm_cash_val,dc_pos_vol,dc_pos_val,dc_ecom_vol,dc_ecom_val,dc_others_vol,dc_others_val,dc_atm_cash_vol,dc_atm_cash_val,dc_pos_cash_vol,dc_pos_cash_val
7,Scheduled Commercial Banks,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Public Sector Banks,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,1,BANK OF BARODA,8407,2621,35956,45227,11954,1642216,2300723,94000243,2286393,"8,363,566.59",2353854,"14,642,361.53",0,0,15840,"73,367.30",4268370,"9,599,386.36",754223,"3,295,029.91",6,13.30,24810278,"119,878,790.05",412,413.01
10,2,BANK OF INDIA,5328,2900,43023,17693,0,991326,74400,47396297,155674,"568,015.77",67924,"312,987.01",0,0,10164,"61,194.09",2944860,"6,054,878.21",930682,"1,592,588.12",0,0,16108478,"65,713,448.55",2554,"2,547.86"
11,3,BANK OF MAHARASHTRA,1977,300,1785,1879,355014,923725,32995,13616903,54016,"233,043.67",20058,"117,289.00",0,0,927,"4,341.64",1360304,"2,341,296.24",421590,"741,243.92",6672,"73,328.26",6128894,"26,622,911.12",0,0
12,4,CANARA BANK,8131,3960,72047,13427,1,2030488,938914,57413339,995475,"3,580,662.33",448596,"2,332,085.56",0,0,92909,"508,696.20",6951817,"17,563,468.81",1319266,"4,269,170.76",3927,"60,718.36",33196988,"156,296,217.28",586,460.80
13,5,CENTRAL BANK OF INDIA,2857,1058,2792,9254,6702,1235590,0,28716820,0,0,0,0,0,0,0,0,1468506,"3,817,061.01",516494,"1,281,249.86",0,0,8801775,"40,814,215.55",4181,"4,361.02"
14,6,INDIAN BANK,4310,599,21791,11074,0,3039310,199110,32192588,158006,"631,446.57",84041,"645,864.80",0,0,4438,"31,526.43",3644297,"8,216,572.77",1075213,"2,331,377.74",3324,"53,738.92",20032288,"92,816,483.13",16480,"16,442.31"


,sr_no,bank_name,atm_onsite,atm_offsite,pos_terminals,micro_atms,bharat_qr,upi_qr,credit_cards,debit_cards,cc_pos_vol,cc_pos_val,cc_ecom_vol,cc_ecom_val,cc_others_vol,cc_others_val,cc_atm_cash_vol,cc_atm_cash_val,dc_pos_vol,dc_pos_val,dc_ecom_vol,dc_ecom_val,dc_others_vol,dc_others_val,dc_atm_cash_vol,dc_atm_cash_val,dc_pos_cash_vol,dc_pos_cash_val
98,Total number of financial transactions done by...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99,Total value of financial transactions done by ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
100,Total number of other financial transactions d...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
101,Total value of other financial transactions do...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
102,Total number of cash withdrawal transactions d...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
103,Total value of cash withdrawal transactions do...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
104,Total number of cash withdrawal transactions d...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
105,Total value of cash withdrawal transactions do...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Observations**
- Bank rows have a number in `sr_no`.
- **Category headings** (e.g. "Public Sector Banks") sit in the `sr_no` column with everything else empty.
- The bottom of the sheet has a **Total** row, then blank rows and footnotes.

## B2. Identify every non-bank row
To separate categories from totals and footnotes, I listed every row whose `sr_no` isn't a number.

In [11]:
is_bank = pd.to_numeric(rbi['sr_no'], errors='coerce').notna()
print("Bank rows:", is_bank.sum())
print("Non-bank rows:", (~is_bank).sum(), "\n")

for idx, row in rbi.loc[~is_bank, ['sr_no']].iterrows():
    print(idx, '|', row['sr_no'])

Bank rows: 65
Non-bank rows: 34 

7 | Scheduled Commercial Banks
8 | Public Sector Banks
21 | Private Sector Banks
43 | Foreign Banks
58 | Payment Banks
65 | Small Finance Banks
78 | Total
79 | The data is provisional
80 | Total number of ATMs & CRMs deployed on-site by the bank
81 | Total number of ATMs & CRMs deployed off-site by the bank
82 | Total number of PoS terminals deployed by the bank
83 | Total number of Micro ATMs deployed by the bank
84 | Total number of Bharat QR Codes deployed by the bank
85 | Total number of UPI QR Codes deployed by the bank
86 | Total number of credit cards issued outstanding (after adjusting the number of cards withdrawan/cancelled)
87 | Total number of debit cards issued outstanding (after adjusting the number of cards withdrawan/cancelled)
88 | Total number of financial transactions done by the credit card issued by the bank at PoS terminals
89 | Total value of financial transactions done by the credit card issued by the bank at PoS terminals
90 | 

**Findings: the non-bank rows fall into three groups**
1. **Category headings:** Public Sector, Private Sector, Foreign, Payment and Small Finance Banks ("Scheduled Commercial Banks" is an umbrella heading over them). → Turn these into a `bank_category` column.
2. **The Total row:** RBI's own sum across all banks. → Keep it separately to **validate** my data.
3. **Footnotes:** blank rows, "the data is provisional", and a definition of each column. The definitions confirm the column map above. → Drop.

## B3. Build a clean table for one month and validate it

In [12]:
categories = ['Public Sector Banks', 'Private Sector Banks', 'Foreign Banks',
              'Payment Banks', 'Small Finance Banks']

# Label heading rows, then forward-fill each label down to the banks beneath it
rbi['bank_category'] = rbi['sr_no'].where(rbi['sr_no'].isin(categories)).ffill()

rbi_total = rbi[rbi['sr_no'] == 'Total'][num_cols].apply(pd.to_numeric, errors='coerce')
banks = rbi[is_bank].copy()
banks[num_cols] = banks[num_cols].apply(pd.to_numeric, errors='coerce')

print(banks['bank_category'].value_counts())

check = pd.DataFrame({'my_sum': banks[num_cols].sum(), 'rbi_total': rbi_total.iloc[0]})
check['difference'] = check['my_sum'] - check['rbi_total']
check

bank_category
Private Sector Banks    21
Foreign Banks           14
Public Sector Banks     12
Small Finance Banks     12
Payment Banks            6
Name: count, dtype: int64


,my_sum,rbi_total,difference
atm_onsite,"126,247.00","126,247.00",0.00
atm_offsite,"93,381.00","93,381.00",0.00
pos_terminals,"8,593,226.00","8,593,226.00",0.00
micro_atms,"1,760,454.00","1,760,454.00",0.00
bharat_qr,"6,003,737.00","6,003,737.00",0.00
upi_qr,"321,379,274.00","321,379,274.00",0.00
credit_cards,"99,500,257.00","99,500,257.00",0.00
debit_cards,"957,565,545.00","957,565,545.00",0.00
cc_pos_vol,"172,558,618.00","172,558,618.00",0.00
cc_pos_val,"585,317,660.76","585,317,660.76",-0.00


**Result:** the bank rows add up to RBI's Total row in every column. The tiny non-zero differences (around 1e-07) are floating-point rounding, not real gaps. This confirms no banks were lost or duplicated and every column is mapped correctly.

## B4. Turn the steps into a reusable function
The same logic has to run on 12 files, so I wrapped it in a function. Two design choices:
- It **searches for the numbered row** instead of assuming a fixed position, in case a file has an extra title line.
- It **validates every file** against RBI's Total row and warns if anything doesn't add up.

In [13]:
def load_rbi_month(path):
    """Load one RBI bank-wise ATM/POS/Card file into a clean bank-level table."""
    raw = pd.read_excel(path, header=None)
    num_row = raw.index[raw.iloc[:, 3] == 1][0]

    df = raw.iloc[num_row + 1:, 1:29].copy()
    df.columns = cols

    df['bank_category'] = df['sr_no'].where(df['sr_no'].isin(categories)).ffill()
    total = df[df['sr_no'] == 'Total']
    df = df[pd.to_numeric(df['sr_no'], errors='coerce').notna()].copy()

    df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')
    total = total[num_cols].apply(pd.to_numeric, errors='coerce')

    df['month'] = pd.to_datetime(path.stem, format='%Y_%m')   # 2024_01 → 2024-01-01

    diff = (df[num_cols].sum() - total.iloc[0]).abs().max()
    if diff > 1:
        print(f"⚠️ {path.name}: totals don't match (max diff {diff:,.0f})")
    else:
        print(f"✓ {path.name}: {len(df)} banks, totals match")
    return df

## B5. Load all 12 months of 2024

In [14]:
all_months = []
for path in sorted(p for p in rbi_dir.iterdir() if p.name.startswith('2024_')):
    try:
        all_months.append(load_rbi_month(path))
    except Exception as e:
        print(f"❌ {path.name}: {e}")

rbi_2024 = pd.concat(all_months, ignore_index=True)
print("\nShape:", rbi_2024.shape)
print("Months:", list(rbi_2024['month'].dt.strftime('%Y-%m').unique()))

✓ 2024_01.XLSX: 65 banks, totals match
✓ 2024_02.XLSX: 65 banks, totals match
✓ 2024_03.XLSX: 65 banks, totals match
✓ 2024_04.XLSX: 64 banks, totals match
✓ 2024_05.XLSX: 64 banks, totals match
✓ 2024_06.XLSX: 64 banks, totals match
✓ 2024_07.XLSX: 64 banks, totals match
✓ 2024_08.XLSX: 64 banks, totals match
✓ 2024_09.XLSX: 64 banks, totals match
✓ 2024_10.XLSX: 64 banks, totals match
✓ 2024_11.XLSX: 64 banks, totals match
✓ 2024_12.XLSX: 64 banks, totals match

Shape: (771, 30)
Months: ['2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12']


**Observations**
- All 12 files loaded and **every file matches RBI's totals**.
- The bank count drops from **65 (Jan–Mar) to 64 (Apr–Dec)**, which needs explaining.

## B6. Check bank names are consistent across months
If a bank is spelled differently in different months, it would be split into two banks. I counted how many months each bank name appears in.

In [15]:
months_per_bank = rbi_2024.groupby('bank_name')['month'].nunique()
print("Unique bank names:", len(months_per_bank))
print("\nBanks not present in all 12 months:")
months_per_bank[months_per_bank < 12].sort_values()

Unique bank names: 66

Banks not present in all 12 months:


bank_name
DHANALAXMI BANK LTD                2
FINCARE SMALL FINANCE BANK LTD     3
DHANALAKSHMI BANK LTD             10
Name: month, dtype: int64

**Findings**
1. **Spelling variation:** "DHANALAXMI BANK LTD" (2 months) + "DHANALAKSHMI BANK LTD" (10 months) = 12 months. It's the same bank, which I standardised to its official name, **Dhanlaxmi Bank**.
2. **A genuine merger:** Fincare Small Finance Bank appears only in Jan–Mar because it **merged into AU Small Finance Bank on 1 April 2024**. That explains the drop from 65 to 64 banks. Kept as is, but **AU Small Finance Bank's jump from April is inorganic growth**, which must be noted in any bank-growth ranking.

In [16]:
rbi_2024['bank_name'] = rbi_2024['bank_name'].replace({
    'DHANALAXMI BANK LTD': 'DHANLAXMI BANK LTD',
    'DHANALAKSHMI BANK LTD': 'DHANLAXMI BANK LTD'
})

months_per_bank = rbi_2024.groupby('bank_name')['month'].nunique()
print("Unique banks after fix:", len(months_per_bank))
print(months_per_bank[months_per_bank < 12])

Unique banks after fix: 65
bank_name
FINCARE SMALL FINANCE BANK LTD    3
Name: month, dtype: int64


## B7. Standardise units, tidy columns and save
- Values are in **Rs '000**, so I converted them to **crore** (÷ 10,000) and added `_cr` to the column names so the unit is visible.
- I dropped `sr_no` (only RBI's row number within each file) and moved the identifier columns to the front.

In [17]:
val_cols = [c for c in rbi_2024.columns if c.endswith('_val')]

rbi_2024[val_cols] = rbi_2024[val_cols] / 10_000
rbi_2024 = rbi_2024.rename(columns={c: c + '_cr' for c in val_cols})

id_cols = ['month', 'bank_category', 'bank_name']
rbi_2024 = rbi_2024[id_cols + [c for c in rbi_2024.columns if c not in id_cols + ['sr_no']]]

rbi_2024.to_csv(processed_dir / 'rbi_bank_cards_2024.csv', index=False)
print("Saved rbi_bank_cards_2024.csv ✓  Shape:", rbi_2024.shape)
rbi_2024.head(3)

Saved rbi_bank_cards_2024.csv ✓  Shape: (771, 29)


,month,bank_category,bank_name,atm_onsite,atm_offsite,pos_terminals,micro_atms,bharat_qr,upi_qr,credit_cards,debit_cards,cc_pos_vol,cc_pos_val_cr,cc_ecom_vol,cc_ecom_val_cr,cc_others_vol,cc_others_val_cr,cc_atm_cash_vol,cc_atm_cash_val_cr,dc_pos_vol,dc_pos_val_cr,dc_ecom_vol,dc_ecom_val_cr,dc_others_vol,dc_others_val_cr,dc_atm_cash_vol,dc_atm_cash_val_cr,dc_pos_cash_vol,dc_pos_cash_val_cr
0,2024-01-01,Public Sector Banks,BANK OF BARODA,8407,"2,621.00",35956,45227,11954,1642216,2300723,94000243,2286393,836.36,2353854,"1,464.24",0,0.00,15840,7.34,4268370,959.94,754223,329.50,6,0.00,24810278,"11,987.88",412,0.04
1,2024-01-01,Public Sector Banks,BANK OF INDIA,5328,"2,900.00",43023,17693,0,991326,74400,47396297,155674,56.80,67924,31.30,0,0.00,10164,6.12,2944860,605.49,930682,159.26,0,0.00,16108478,"6,571.34",2554,0.25
2,2024-01-01,Public Sector Banks,BANK OF MAHARASHTRA,1977,300.00,1785,1879,355014,923725,32995,13616903,54016,23.30,20058,11.73,0,0.00,927,0.43,1360304,234.13,421590,74.12,6672,7.33,6128894,"2,662.29",0,0.00


---
# Summary of cleaning decisions

| Issue | Source | Decision |
|---|---|---|
| Thousand separators in volumes | NPCI | Parsed with `thousands=','` |
| Stray `Unnamed: 0` index column | NPCI (AePS – Funds Transfer) | Dropped; columns renamed by name |
| `24-Jan` month format | NPCI | Parsed with `format='%y-%b'` |
| CTS and NACH – Debit identical in all months | NPCI | Both excluded (data error) |
| Merged multi-row headers | RBI | Mapped columns using RBI's numbered header row |
| Category headings mixed with bank rows | RBI | Converted to a `bank_category` column |
| Total row and footnotes | RBI | Total used for validation, then removed |
| Two spellings of Dhanlaxmi Bank | RBI | Standardised |
| Fincare SFB merged into AU SFB (Apr 2024) | RBI | Kept; AU's growth from April flagged as inorganic |
| Values in Rs '000 | RBI | Converted to crore (`_cr` columns) |

**Next:** Stage 2, SQL analysis on the two processed tables.

In [18]:
missing = rbi_2024.isna().sum()
print(missing[missing > 0])
display(rbi_2024[rbi_2024['atm_offsite'].isna()][['month', 'bank_name', 'atm_onsite', 'atm_offsite']])

atm_offsite    2
dtype: int64


,month,bank_name,atm_onsite,atm_offsite
385,2024-06-01,UNITY SMALL FINANCE BANK LTD,232,NaN
449,2024-07-01,UNITY SMALL FINANCE BANK LTD,232,NaN


In [19]:
gitignore = """.ipynb_checkpoints/
__pycache__/
*.pyc
.vscode/
~$*
"""
(project / '.gitignore').write_text(gitignore, encoding='utf-8')
print("Created .gitignore ✓")

Created .gitignore ✓
